In [1]:
import pandas as pd
import numpy as np
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

In [2]:
distance_matrix = np.load('../data/distance_matrix.npy')
sample = pd.read_csv('../data/sample_stops.csv')

BIG_M = 999_999_999
n_unreachable = int((distance_matrix >= BIG_M).sum())
print(f"unreachable pairs: {n_unreachable}")
assert n_unreachable == 0, "some stop pairs have no driving path — graph needs fixing"

unreachable pairs: 0


In [3]:
distance_matrix = distance_matrix.astype(int)
manager = pywrapcp.RoutingIndexManager(
    len(distance_matrix), #number of locations
    3,                    #number of vehicles
    0                     #depot index
)
routing = pywrapcp.RoutingModel(manager)

In [4]:
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

In [5]:
transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

In [6]:
print(len(sample))

21


In [7]:
rng = np.random.default_rng(42)
sample['demand'] = rng.integers(1, 6, size=len(sample))
sample.loc[0, 'demand'] = 0          # depot ships nothing
sample.to_csv('../data/sample_stops.csv', index=False)
print(f"total demand: {sample['demand'].sum()} across {len(sample)-1} stops")

total demand: 66 across 20 stops


In [8]:
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return int(sample.iloc[from_node]['demand'])
demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

In [9]:
print(sample['demand'].sum())

66


In [10]:
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index, 
    0,
    [25, 25, 25],
    True,
    'Capacity'
)

True

In [11]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 30

In [12]:
solution = routing.SolveWithParameters(search_parameters)

In [13]:
if solution:
    print(solution)
else:
    print("No solution found")

Assignment(Capacity0 (0) | Capacity1 (17) | Capacity2 (9) | Capacity3 (6) | Capacity4 (0) | Capacity5 (4) | Capacity6 (13) | Capacity7 (13) | Capacity8 (4) | Capacity9 (0) | Capacity10 (22) | Capacity11 (7) | Capacity12 (14) | Capacity13 (18) | Capacity14 (3) | Capacity15 (9) | Capacity16 (1) | Capacity17 (0) | Capacity18 (12) | Capacity19 (1) | Capacity20 (21) | Capacity21 (0) | Capacity22 (0) | Capacity23 (24) | Capacity24 (25) | Capacity25 (17) | Nexts0 (17) | Nexts1 (20) | Nexts2 (7) | Nexts3 (2) | Nexts4 (14) | Nexts5 (15) | Nexts6 (12) | Nexts7 (1) | Nexts8 (3) | Nexts9 (19) | Nexts10 (24) | Nexts11 (18) | Nexts12 (13) | Nexts13 (10) | Nexts14 (11) | Nexts15 (6) | Nexts16 (8) | Nexts17 (16) | Nexts18 (25) | Nexts19 (5) | Nexts20 (23) | Nexts21 (9) | Nexts22 (4) | Active0 (1) | Active1 (1) | Active2 (1) | Active3 (1) | Active4 (1) | Active5 (1) | Active6 (1) | Active7 (1) | Active8 (1) | Active9 (1) | Active10 (1) | Active11 (1) | Active12 (1) | Active13 (1) | Active14 (1) | Activ

In [14]:
total_distance = 0

for vehicle_id in range(3):
    index = routing.Start(vehicle_id)
    route, route_distance = [], 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(sample.iloc[node]['Address'])
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += distance_matrix[
            manager.IndexToNode(previous_index)
        ][manager.IndexToNode(index)]

    route.append(sample.iloc[manager.IndexToNode(index)]['Address'])
    total_distance += route_distance

    print(f"Driver {vehicle_id + 1}:")
    for stop in route:
        print(f"  -> {stop}")
    print(f"  Route distance: {round(route_distance/1000, 2)} km\n")

print(f"FLEET TOTAL: {round(total_distance/1000, 2)} km")

Driver 1:
  -> 4520 Bullock Farm Rd
  -> 4325 Glenwood Ave UNIT B
  -> 4131 Parklake Ave STE 200
  -> 6308 Angus Dr STE E
  -> 9400 Brier Creek Pkwy STE 203
  -> 7201 Creedmoor Rd STE 150
  -> 6837 Falls Of Neuse Rd STE 206
  -> 6675 Falls Of Neuse Rd STE 115
  -> 5811 Poyner Village Pkwy
  -> 4520 Bullock Farm Rd
  Route distance: 68.93 km

Driver 2:
  -> 4520 Bullock Farm Rd
  -> 111 E North St
  -> 4200 Hillsborough St
  -> 3214 Student Ln
  -> 701 Corporate Center Dr STE 185
  -> 6303 Chapel Hill Rd
  -> 547 Pylon Dr
  -> 1010 Main Campus Dr STE 150
  -> 226 E Martin St
  -> 4520 Bullock Farm Rd
  Route distance: 45.63 km

Driver 3:
  -> 4520 Bullock Farm Rd
  -> 3100 Highwoods Blvd STE 115
  -> 3757 Benson Dr
  -> 6411 Lakecrest Dr
  -> 3225 Capital Blvd
  -> 4520 Bullock Farm Rd
  Route distance: 30.97 km

FLEET TOTAL: 145.53 km


In [15]:
results = []

for vehicle_id in range(3):
    index = routing.Start(vehicle_id)
    stop_number = 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        results.append({
            'vehicle': vehicle_id + 1,
            'stop_sequence': stop_number,
            'node': node,
            'address': sample.iloc[node]['Address'],
            'latitude': sample.iloc[node]['latitude'],
            'longitude': sample.iloc[node]['longitude'],
        })
        stop_number += 1
        index = solution.Value(routing.NextVar(index))

    # the loop exits AT the end node, so append it explicitly
    node = manager.IndexToNode(index)
    results.append({
        'vehicle': vehicle_id + 1,
        'stop_sequence': stop_number,
        'node': node,
        'address': sample.iloc[node]['Address'],
        'latitude': sample.iloc[node]['latitude'],
        'longitude': sample.iloc[node]['longitude'],
    })

results_df = pd.DataFrame(results)
results_df.to_csv('../data/solution_routes.csv', index=False)
print(f"{len(results_df)} rows written")

26 rows written
